In [4]:
from google.colab import files
uploaded = files.upload()


import pandas as pd
import numpy as np

# Load WITHOUT header
df = pd.read_csv("BreastCancerWc.csv", header=None)

# Assign correct column names
df.columns = [
    'id', 'clump_thickness', 'cell_size', 'cell_shape',
    'marginal_adhesion', 'epithelial_cell_size',
    'bare_nuclei', 'chromatin', 'nucleoli',
    'mitoses', 'class'
]

print(df.head())
print(df.info())

Saving BreastCancerWc.csv to BreastCancerWc.csv
        id  clump_thickness  cell_size  cell_shape  marginal_adhesion  \
0  1000025                5          1           1                  1   
1  1002945                5          4           4                  5   
2  1015425                3          1           1                  1   
3  1016277                6          8           8                  1   
4  1017023                4          1           1                  3   

   epithelial_cell_size bare_nuclei  chromatin  nucleoli  mitoses  class  
0                     2           1          3         1        1      2  
1                     7          10          3         2        1      2  
2                     2           2          3         1        1      2  
3                     3           4          3         7        1      2  
4                     2           1          3         1        1      2  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 

In [5]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Convert to numeric
df = df.apply(pd.to_numeric, errors='coerce')

# Drop missing values
df.dropna(inplace=True)

# Remove negative values
df = df[(df >= 0).all(axis=1)]

# Drop ID column (not useful)
df.drop('id', axis=1, inplace=True)

print("After Cleaning:", df.shape)

After Cleaning: (683, 10)


In [6]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

In [7]:
from sklearn.preprocessing import StandardScaler

# Target
#Original    	     Converted
#2 → Benign     	0 → Benign
#4 → Malignant	  1 → Malignant
y = df['class'].apply(lambda x: 1 if x == 4 else 0)

# Features
X = df.drop('class', axis=1)

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test))

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
acc_nb = accuracy_score(y_test, nb.predict(X_test))

print("Logistic Regression:", acc_lr)
print("Naive Bayes:", acc_nb)

Logistic Regression: 0.9696969696969697
Naive Bayes: 0.9494949494949495


In [9]:
#compare
if acc_lr > acc_nb:
    print("Logistic Regression is better")
else:
    print("Naive Bayes is better")

Logistic Regression is better
